# Explore and train an Iris classifier

Start Jupyter with `uv run jupyter lab notebooks/iris.ipynb` from the `python/` folder.
Run the cells from top to bottom. We reuse the functions in `src/iris_mlops/train.py` so the
notebook and command-line script follow the same training process.


In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.datasets import load_iris
from sklearn.metrics import ConfusionMatrixDisplay

from iris_mlops.train import split_data, train_model, evaluate_model, save_artifacts


## 1. Meet the dataset

Each row describes a flower using four measurements in centimeters. The target
is one of three species. The data comes bundled with scikit-learn.


In [ ]:
iris = load_iris(as_frame=True)
flowers = iris.data.copy()
flowers["species"] = iris.target.map(dict(enumerate(iris.target_names)))
display(flowers.head())
display(flowers.describe())
display(flowers["species"].value_counts().rename("flowers"))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for species, group in flowers.groupby("species"):
    ax.scatter(group["petal length (cm)"], group["petal width (cm)"], label=species)
ax.set(xlabel="Petal length (cm)", ylabel="Petal width (cm)", title="Iris petal measurements")
ax.legend()
plt.show()


## 2. Split and train

Keep 20% of rows for testing and preserve the class proportions. The scaler and
classifier learn from the training rows only. Start with the default settings;
smaller `C` means stronger regularization. Use cross-validation on the training
set if you later want to select settings, keeping the test set for a final check.


In [ ]:
TEST_SIZE = 0.2
RANDOM_STATE = 42
C = 1.0

X_train, X_test, y_train, y_test = split_data(iris, TEST_SIZE, RANDOM_STATE)
model = train_model(X_train, y_train, C=C)
print(f"Training rows: {len(X_train)} | Test rows: {len(X_test)}")
model


## 3. Evaluate on unseen rows

Accuracy is the fraction of correct predictions. The table shows per-species
precision, recall, F1, and support (number of test examples). In the confusion
matrix, rows are actual species and columns are predicted species.


In [ ]:
metrics = evaluate_model(model, X_test, y_test, iris.target_names)
print(f"Test accuracy: {metrics['accuracy']:.1%}")
display(pd.DataFrame({name: metrics[name] for name in iris.target_names}).T)

ConfusionMatrixDisplay.from_estimator(
    model, X_test, y_test, display_labels=iris.target_names, cmap="Blues"
)
plt.title("Test-set predictions")
plt.tight_layout()
plt.show()


## 4. Inspect individual predictions

Compare the predicted and actual species. An empty mistakes table means every
test row was classified correctly for this split; it does not guarantee perfect
performance on future data.


In [ ]:
predictions = model.predict(X_test)
results = X_test.copy()
results["actual"] = iris.target_names[y_test.to_numpy()]
results["predicted"] = iris.target_names[predictions]
display(results.head(10))
display(results.loc[results["actual"] != results["predicted"]])


## 5. Predict one flower

Edit these measurements in centimeters, keeping the feature order:
sepal length, sepal width, petal length, petal width.


In [ ]:
sample = pd.DataFrame([[5.1, 3.5, 1.4, 0.2]], columns=iris.feature_names)
print("Predicted species:", iris.target_names[model.predict(sample)[0]])
pd.Series(model.predict_proba(sample)[0], index=iris.target_names, name="probability")


## 6. Save this model (optional)

Uncomment the line below to save the fitted pipeline and metrics to `artifacts/`.
This replaces any artifacts previously created by `uv run python -m iris_mlops.train`.


In [ ]:
# save_artifacts(model, metrics)
